# 크롤링
크롤링은 웹의 요소들을 긁어오는 일이라고 할 수 있습니다.

일반적으로 웹 서비스는 서버에서 클라이언트로 정적 요소들을 보내어 동적으로 보여주는 방식입니다.

예를 들어서 과정을 설명해보면
1. 사용자가 `www.example.com`에 간다면 해당 도메인(IP)에 기본 데이터를 요청하게 됩니다. 
2. 해당 도메인에서 그 요청에 대한 정적 데이터들을 전송해주게 됩니다. 대표적으로 `index.html`, `script.js`, `style.css`가 존재합니다.
3. 정적 페이지 또는 서버사이드 렌더링의 경우에는 그대로 `html`에 스타일만 입혀져서 보여주게 됩니다.
4.  동적 페이지의 경우에는 `.js` 파일을 통해 페이지의 요소가 채워지게 됩니다.

위의 과정에서 3번과 4번을 나누어 설명한 이유는 이번에 알려줄 정적/동적 웹 크롤링의 차이를 나타내기 위함입니다.

예를 들어서 바나프레소의 사이트를 이용해 보겠습니다.

In [ ]:
import requests 

def get_html_text(url):
  return requests.request("get", url).text

In [26]:
from pathlib import Path

url = "https://www.banapresso.com/"

html = get_html_text(url)

# 네이버
# html = get_html_text("https://www.naver.com/")

CURR_DIR = Path()


with open(CURR_DIR / "bana.html", "w", encoding="utf-8") as f:
  f.write(html)

html = html.split('\n')

# 20행만 출력
for h in html[:20]:
  print(h)


<!doctype html>
<html lang="ko">
  <head>
    <meta charset="utf-8" />
    <meta name="viewport" content="user-scalable=no, width=device-width" />
    <meta name="format-detection" content="telephone=no" />
    <meta http-equiv="Cache-Control" content="no-Cache" />
    <meta http-equiv="Pragma" content="no-cache" />
    <meta http-equiv="Expires" content="-1" />
    <meta name="theme-color" content="#fff" />
    <link rel="shortcut icon" href="/ico_logo.ico" />
    <link rel="manifest" href="/manifest.json" />

    <!-- CSS, Font -->
    <link
      rel="stylesheet preload"
      href="https://fonts.googleapis.com/earlyaccess/notosanskr.css"
      as="style"
      crossorigin="anonymous"
    />


# 결과
<img src="https://raw.githubusercontent.com/hurwan0629/lang-chain-KDT/c64f929b41c49a23b97192593debf73b66351b06/workspace/personal/pure_python/05_selenium/image.png" width="500px">

이와 같이 크롤링은 잘 되었지만 동적 렌더링을 하는 바나프레소의 성격 때문에 문제가 발생함을 알 수 있습니다.

# 동적 크롤링
이를 해결하기 위해서 몇가지 동적 크롤링 방식을 사용할 수 있습니다.

### 라이브러리
- `selenium`: 오래된 표준 동적 크롤링 도구로 `WebDriver`을 기반으로 동작합니다. 오래된 자료가 많고 `W3C` 표준으로 지정되어있습니다.
- `playwright`: 비교적 최신 도구로 `js` 렌더링 등에 대한 기다리는 기능이 더욱 강화되어있습니다.
- 그 외에도 Puppeteer, pyppeteer, nodriver, undetected-chromedriver(봇 방지 회피 셀레니움) 등이 존재합니다.

이번에는 `selenium`을 먼저 사용해보며 예시를 보겠습니다.

# 셀레니움
셀레니움의 구성 요소에 대해서 알아보겠습니다.
### 시작
셀레니움 라이브러리는 콘솔에 `pip install selenium`을 통해 받을 수 있으며, 

## WebDriver
셀레니움의 핵심 객체로 브라우저를 직접 다루는 주체입니다. 아래와 같이 사용 가능합니다.

구조는 `Python Selenium 코드 -> 브라우저Driver -> 브라우저`의 형태를 가지게 됩니다.

- `find_element(By.?, "선택자")`: 요소를 찾아줍니다.
- `find_elements(By.?, "선택자")`: 요소를 리스트로 반환합니다.

In [48]:
from selenium import webdriver
import time

def go_and_close_dec(func):
  def wrapper(*args, **wargs):

    # webdriver.브라우저명()으로 객체 생성
    driver = webdriver.Chrome()

    driver.get(url)

    # # 뒤로 가기
    # driver.back()
    # # 앞으로 가기
    # driver.forward()
    # # 새로고침
    # driver.refresh()
    # # 나가기

    result = func(driver, *args, **wargs)

    # 1초 뒤에 종료
    time.sleep(1)
    driver.quit()
    
    return result

  return wrapper

@go_and_close_dec
def nothing(driver):
  return

nothing()

# 결과
<img src="https://raw.githubusercontent.com/hurwan0629/lang-chain-KDT/c64f929b41c49a23b97192593debf73b66351b06/workspace/personal/pure_python/05_selenium/image-1.png" width="500px">

와 같이 동적으로 `js` 로딩까지 되는 것을 확인할 수 있습니다.

# WebElement
셀레니움이 `DOM` 요소를 다루는 주요 방식으로 많은 셀레니움 탐색 메서드는 `WebElement`를 반환합니다.

### 메서드
- `click()`: 해당 요소를 클릭합니다
- `send_keys("입력할 문자열")`: 키보드 입력을 시뮬레이션하빈다.
- `clear()`: 입력 필드를 지웁니다.
- `text`: 요소 내부의 텍스트를 반환합니다.
- `get_attribute("속성 명")`: 지정한 HTML 속성값을 반환합니다.
- `value_of_css_property(속성 명)`: 요소에 적용된 속성값을 적용합니다.
- `rect`/`size`/`location`: 요소의 크기 및 화면상 좌측 상단 기준의 좌표값을 반환합니다.

> 찾는 요소가 없으면 `selenium.common.exceptions`의 `NoSuchElementException` 에러가 발생합니다

# By
여러 요소 탐색 방식을 지원하기 위해 `selenium.webdriver.common.by` 에 존재하는 객체입니다.
- `CSS_SELECTOR`: 일반적인 탐색 방식입니다.
- `XPATH`: XML이나 HTML 문서의 특정 요소에 접근하기 위한 절대 경로 언어입니다.
- `ID`: `id`를 통해 요소를 찾습니다. (빠릅니다.)
- `NAME`: `name` 속성값을 이용합니다.
- `TAG_NAME`: `div`, `a`, `p` 같은 HTML 태그를 이용하여 요소를 찾습니다.

In [49]:

from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

@go_and_close_dec
def desc_web_element_and_by(driver):
  bana_logo = None
  try:
    bana_logo = driver.find_element(
      By.CSS_SELECTOR, "h1.logo > a > img"
    )
    print(type(bana_logo))
  except NoSuchElementException as e:
    print("요소 없음 에러 발생")
  except Exception as e:
    print(e)
  print(bana_logo)
  return bana_logo.get_attribute("src")

bana_logo = desc_web_element_and_by()

<class 'selenium.webdriver.remote.webelement.WebElement'>
<selenium.webdriver.remote.webelement.WebElement (session="5ff8b6094148c856704f37974465c022", element="f.2AB18627888390C52EE7B513C6C1EA79.d.C512C320126F8DB7AB9F4A45BDA6E10B.e.10")>


In [54]:
# 뽑은 이미지 보기
from IPython.display import Image, display

res = requests.request("get", bana_logo)
print(res.status_code)
print(res.headers.get("Content-Type"))

display(Image(data=res.content))

200
image/svg+xml


# 파이썬으로 출력이 안되어서 직접 저장
<img src="https://raw.githubusercontent.com/hurwan0629/lang-chain-KDT/c64f929b41c49a23b97192593debf73b66351b06/workspace/personal/pure_python/05_selenium/output.svg" width="500px">

# Wait
동적 페이지에서 HTMl이 바로 뜨지 않을경우를 고려하여 기다리는 것을 도와주는 메서드입니다.

## `WebDriverWait`
`selenium.webdriver.support.ui`에서 `import` 할 수 있으며 지정한 시간동안 동적으로 기다려주는 명시적 대기를 도와줍니다. 네트워크 지연 또는 비동기 로딩으로 인해 코드가 웹 요소를 찾지 못해 발생하는 `NoSuchElementException` 에러를 방지하는 데 필수적입니다.

이를 사용하려면 **대기 조건**들과 함께 호출해야합니다.

대기 조건은 `selenium.webdriver.support`의 `expected_conditions`에 존재합니다.

만약 시간이 초과된다면 `TimeoutException` 가 일어나게 됩니다.

## EC 조건 종류
#### 요소의 존재/시각적 상태
- `presence_of_element_located`: 요소가 DOM 내에 존재할 때 반환 (화면에 안 보여도 됨)
- `visibility_of_element_located`: 요소가 DOM에 존재하고 화면에도 실제로 보일 때 반환
- `invisibility_of_element_located`: 요소가 화면에서 사라지거나 DOM에서 제거될 때까지 대기
- `presence_of_all_elements_located`: 조건에 맞는 요소가 최소 1개 이상 DOM에 생성되면 리스트 반환
- `visibility_of_any_elements_located`: 조건에 맞는 요소 중 최소 1개 이상이 화면에 보이면 리스트 반환
#### 상호작용 가능 상태
- `element_to_be_clickable`: 요소가 화면에 보이고 활성화되어 클릭할 수 있을 때 반환
- `element_to_be_selected`: 지정한 요소(체크박스, 라디오 등)가 선택된 상태일 때 만족
- `element_selection_state_to_be`: 요소의 선택 여부(True/False) 상태를 직접 지정해 확인
#### 텍스트 확인
- `text_to_be_present_in_element`: 지정한 요소 안에 특정 문자열이 포함되어 있는지 확인
- `text_to_be_present_in_element_value`: input 창 등의 value 속성값 안에 특정 문자열이 포함되어 있는지 확인
#### 브라우저 창 및 프레임
- `title_is`: 브라우저 탭의 제목이 지정한 문자열과 정확히 일치할 때 만족
- `title_contains`: 브라우저 탭 제목에 특정 문자열이 포함되어 있을 때 만족
- `url_to_be`: 현재 브라우저의 URL 주소가 지정한 주소와 정확히 일치할 때 만족
- `url_contains`: 현재 URL 주소에 특정 문자열이 포함되어 있는지 확인
- `frame_to_be_available_and_switch_to_it`: 해당 iFrame으로 전환할 수 있는 상태가 되면 자동 전환
- `alert_is_present`: 화면에 경고창(Alert) 팝업이 나타났는지 확인 후 객체 반환
#### 조건 논리 조합
- `any_of`: 전달된 여러 조건 중 어느 하나라도 만족하면 대기 종료 (OR)
- `all_of`: 전달된 여러 조건이 모두 만족할 때까지 대기 (AND)
- `none_of`: 전달된 여러 조건이 모두 만족하지 않을 때까지 대기 (NOT)

In [60]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

@go_and_close_dec
def wait_example(driver):
  # 10초동안 명시적으로 기다리는 드라이버 생성
  wait = WebDriverWait(driver, 10)

  try:
    search_box = wait.until(
        # EC.presence_of_element_located((By.ID, "asdf")) # 없으면 10초 후 TimeoutException 반환
        EC.presence_of_element_located((By.ID, "contents")) # 있으면 바로 반환
    )
    print(search_box)
  except TimeoutException as te:
    print("해당 요소가 존재하지 않습니다.")

wait_example()

<selenium.webdriver.remote.webelement.WebElement (session="7536a9e561c50529beea51a2d95c777f", element="f.8DCEC50CBF432CF092963D481E2EFE2E.d.B9DC6BCCD49DBC57E63EF0694E709329.e.32")>


# Options
브라우저 실행 옵션을 설정하는 객체입니다. 
`옵션객체.add_arguments(옵션)`을 통해 옵션을 만들어 `webdriver(options=옵션객체)`로 설정이 가능합니다. 예를 들어 다음과 같은 설정을 할 수 있습니다.
#### 브라우저 설정 옵션
- `options.add_argument("--start-maximized")`: 창 최대화
- `options.add_argument("--headless=new")`: 브라우저 열지 않고 크롤링 (사용자)
- `options.add_argument("--window-size=1920,1080")`: 윈도우 크기 구체적으로 지정
- `options.add_argument("--incognito")`: 시크릿 모드
- `options.add_argument("--disable-gpu")`: gpu가속 비활성화 (EC2같은 GPU 없는 환경에서 쓰임)
- `options.add_argument("--lang=ko-KR")`: 사용 언어 지정
- `options.add_argument("--disable-notifications")`: 알림 차단
- `options.add_argument("--disable-popup-blocking")`: 팝업 차단
- `options.add_argument("--user-agent=User-Agent")`: 에이전트 설정
- `options.add_argument("--user-data-dir=C:/selenium_profile")`: 사용자 디렉터리 지정
#### 브라우저 권한과 설정
- `options.add_experimental_option(설정 종류, 설정에 대한 인자)`
  - `useAutomationExtension`: `False`를 통해 봇 탐지 당하는 것을 방지하기 위해 확장 프로그램 기능을 끕니다.
  - `excludeSwitches`: `["enable-automation"]`를 통해 상단의 자동화 제어 문구를 제거할 수 있습니다.
  - `prefs`: 2번 인자로 허가 설정을 딕셔너리로 넣습니다. 권한의 경우 `1`은 허용, `2`는 차단, `0`은 기본값입니다.
    - `profile.default_content_setting_values.geolocation`: 위치 권한
    - `profile.default_content_setting_values.notifications`: 알림 권한
    - `profile.managed_default_content_settings.images`: 이미지 로딩
    - `download.default_directory`: 기본 경로 지정
    - `download.prompt_for_download`: 다운로드 확인 팝업 생략
#### 확장 프로그램
- `options.add_extension("extension.crx")`: 확장 파일을 넣을 수 있습니다.
#### 브라우저 경로 지정
- `options.binary_location`: 크롬 실행 위치를 직접 지정 가능합니다.

# Service
드라이버의 실행을 설정합니다. (`Options`는 브라우저를 설정합니다. `WebDriver`은 브라우저 조작 객체입니다.)

자주 쓰이지 않기 때문에 언급만 하고 넘어가겠습니다.

# Keys
키보드 입력을 흉내내는 객체입니다.

찾은 `WebElement`객체에 `.send_keys("넣을 문자열")` 또는 `.send_keys(Keys.ENTER)` 등과 같이 사용할 수 있습니다. `Ctrl`과 조합할 경우에는 `search_box.send_keys(Keys.CONTROL, "c")`과 같이 인자 2개로 사용 가능합니다.

# ActionChains
액션 체인은 마우스와 키보드의 복합 동작을 순서대로 쌓아서 실행하는 객체입니다.

`selenium.webdriver.common.action_chains`의 `ActionChains`에서 꺼내서 사용이 가능하며

`actions = ActionChains(driver)` 함수형 프로그래밍 형식으로 사용 가능합니다.

### 체이닝 메서드 종류
- `perform()`: 최종 입력 메서드입니다. 실행하려면 반드시 작성해야합니다.
- `click()`: 클릭입니다.
- `move_to_element(WebElement객체)`: 요소의 정중앙으로 이동합니다.
- `move_by_offset(x이동픽셀, y이동픽셀)`: 마우스 위치를 이동합니다.
- `double_click(WebElement객체)`: 더블클릭합니다.
- `drag_and_drop(WebElement객체, 목표점)`: 객체를 목표까지 이동시킵니다.
- `scroll_by_amount(X 스크롤 양, Y 스크롤 양)`: 마우스 위치에서 스크롤 합니다.
- 또한 키보다 등의 다른 동작도 함께 사용 가능합니다.

# JavaScript Executor
셀레니움은 자바 스크립트를 실행해주는 기능 또한 제공해주고 있습니다.

`driver.execute_script("실행할 JS 코드")`를 통해서 실행할 수 있으며 두번째 인자에 객체를 넣어서 `arguments[0]`과 같이 그 객체를 `DOM` 요소처럼 사용할 수 있습니다.

# 직접 해보기
[바나프레소 매장들 뽑아보기](../../assignment/07_bana_selenium/main.ipynb)